<a href="https://colab.research.google.com/github/nishithy13/ds2002-fa26/blob/main/notebooks/01-foundations/2026-10-02%20%E2%80%94%20Normalize_and_Join_Nested_JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** order line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])

lines = pd.json_normalize(
    orders_json,
    record_path = 'lines',
    meta = ['order', 'vendor_id'])

lines

# TODO: assert the row count and the quantity total

assert len(lines) == 4
assert lines['qty'].sum() == 7

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO

vendors = pd.json_normalize(vendors_json)

joined = lines.merge(
    vendors,
    on = 'vendor_id',
    how = 'left',
    indicator = True)

joined

joined[joined['_merge'] == 'left_only']

,item,qty,order,vendor_id,name,zone,_merge


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO

totals = joined.groupby('name')['qty'].sum()

totals

# Both vendors matched so that is why there is no unmatched vendor in the table.

,qty
name,
Cav Merch,1
Hoos Burgers,6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO

zone_totals = joined.groupby('zone')['qty'].sum()

zone_totals

print('Zone with most items:', zone_totals.idxmax())
print('Number of items:', zone_totals.max())


Zone with most items: A
Number of items: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO

ragged_with_lines = [x for x in ragged_json if 'lines' in x]

ragged_lines = pd.json_normalize(
    ragged_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id'])

ragged_lines

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item makes it easy to calculate the total quantity of each item or vendor. With one row per order, the items would be grouped inside each other. This makes it harder to calcualte the totals.

b) A missing quantity means that there is no answer to how many were sold. On the other hand, a quantity of zero means there was none sold. If missing quantities are treated as zero, the totals could be wrong due to interpreting missing data as no items being sold.